<a href="https://colab.research.google.com/github/jesuscarpio5/Analisis-deportivo/blob/main/01_jugadores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install mplsoccer statsbombpy

In [20]:
from statsbombpy import sb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import warnings
warnings.filterwarnings("ignore")

partidos = sb.matches(competition_id=43, season_id=3)

lista = []
for m in partidos.match_id:
    e = sb.events(match_id=m)
    e["match_id"] = m
    lista.append(e)

eventos_todos = pd.concat(lista, ignore_index=True)

# REGLA: sin tanda de penaltis (period 5). Solo periodos 1-4 (90' + prórroga)
eventos = eventos_todos[eventos_todos.period < 5].copy()
print(len(partidos), "partidos,", len(eventos), "eventos")

64 partidos, 227731 eventos


In [21]:
minutos = {}
for m in partidos.match_id:
    ev = eventos[eventos.match_id == m]
    fin = max(ev.minute.max(), 90)   # el minuto es acumulado: 120 si hubo prórroga

    for _, t in ev[ev.type == "Starting XI"].iterrows():
        for j in t.tactics["lineup"]:
            k = (j["player"]["name"], t.team)
            minutos[k] = minutos.get(k, 0) + fin

    for _, s in ev[ev.type == "Substitution"].iterrows():
        sale, entra, minuto = s.player, s.substitution_replacement, s.minute
        minutos[(sale, s.team)] = minutos.get((sale, s.team), 0) - (fin - minuto)
        minutos[(entra, s.team)] = minutos.get((entra, s.team), 0) + (fin - minuto)

df_min = (pd.Series(minutos).rename("minutos").reset_index()
          .rename(columns={"level_0": "jugador", "level_1": "equipo"}))
df_min.sort_values("minutos", ascending=False).head(10)

,jugador,equipo,minutos
42,Jordan Pickford,England,714
283,Luka Modrić,Croatia,706
278,Dejan Lovren,Croatia,674
45,Harry Maguire,England,669
44,John Stones,England,666
284,Ivan Rakitić,Croatia,663
239,Raphaël Varane,France,662
11,Thibaut Courtois,Belgium,659
276,Danijel Subašić,Croatia,649
279,Domagoj Vida,Croatia,649


In [22]:
ev = eventos.dropna(subset=["player"]).copy()

# Pases progresivos
p = ev[(ev.type == "Pass") & (ev.pass_outcome.isna())].dropna(subset=["pass_end_location"]).copy()
p[["x","y"]] = pd.DataFrame(p.location.tolist(), index=p.index)
p[["x2","y2"]] = pd.DataFrame(p.pass_end_location.tolist(), index=p.index)
d1 = np.sqrt((120-p.x)**2 + (40-p.y)**2)
d2 = np.sqrt((120-p.x2)**2 + (40-p.y2)**2)
prog = p[(d2 < 0.75*d1) & (p.x2 > p.x)]

tiros = ev[ev.type == "Shot"]

tabla = pd.DataFrame({
    "pases":          p.groupby(["player","team"]).size(),
    "progresivos":    prog.groupby(["player","team"]).size(),
    "tiros":          tiros.groupby(["player","team"]).size(),
    "goles":          tiros[tiros.shot_outcome == "Goal"].groupby(["player","team"]).size(),
    "xg":             tiros.groupby(["player","team"]).shot_statsbomb_xg.sum(),
    "regates":        ev[(ev.type == "Dribble") & (ev.dribble_outcome == "Complete")].groupby(["player","team"]).size(),
    "recuperaciones": ev[ev.type.isin(["Ball Recovery","Interception"])].groupby(["player","team"]).size(),
}).fillna(0).reset_index().rename(columns={"player":"jugador","team":"equipo"})

tabla = tabla.merge(df_min, on=["jugador","equipo"], how="left")
tabla = tabla[tabla.minutos >= 180].copy()   # al menos 2 partidos completos

for c in ["pases","progresivos","tiros","goles","xg","regates","recuperaciones"]:
    tabla[c + "_90"] = tabla[c] / tabla.minutos * 90

tabla.sort_values("progresivos_90", ascending=False).head(15)

,jugador,equipo,pases,progresivos,tiros,goles,xg,regates,recuperaciones,minutos,pases_90,progresivos_90,tiros_90,goles_90,xg_90,regates_90,recuperaciones_90
288,Jérôme Boateng,Germany,162,26.0,3.0,0.0,0.041786,0.0,11.0,190,76.736842,12.315789,1.421053,0.000000,0.019794,0.000000,5.210526
78,Branislav Ivanović,Serbia,76,25.0,0.0,0.0,0.000000,2.0,7.0,192,35.625000,11.718750,0.000000,0.000000,0.000000,0.937500,3.281250
542,Toni Kroos,Germany,292,36.0,16.0,1.0,0.693363,5.0,25.0,289,90.934256,11.211073,4.982699,0.311419,0.215926,1.557093,7.785467
341,Marcelo Vieira da Silva Júnior,Brazil,238,33.0,4.0,0.0,0.059038,2.0,29.0,298,71.879195,9.966443,1.208054,0.000000,0.017830,0.604027,8.758389
268,Josh Risdon,Australia,113,30.0,1.0,0.0,0.044452,5.0,11.0,282,36.063830,9.574468,0.319149,0.000000,0.014187,1.595745,3.510638
389,Mohamed Abdul Shafy Sayed Abouezeid,Egypt,136,28.0,0.0,0.0,0.000000,4.0,22.0,285,42.947368,8.842105,0.000000,0.000000,0.000000,1.263158,6.947368
544,Trent Sainsbury,Australia,169,27.0,3.0,0.0,0.307944,1.0,13.0,282,53.936170,8.617021,0.957447,0.000000,0.098280,0.319149,4.148936
296,Kasper Schmeichel,Denmark,67,38.0,0.0,0.0,0.000000,0.0,18.0,402,15.000000,8.507463,0.000000,0.000000,0.000000,0.000000,4.029851
586,Yuri Zhirkov,Russia,67,21.0,2.0,0.0,0.032563,1.0,7.0,224,26.919643,8.437500,0.803571,0.000000,0.013083,0.401786,2.812500
382,Miguel Ángel Trauco Saavedra,Peru,156,26.0,0.0,0.0,0.000000,2.0,21.0,283,49.611307,8.268551,0.000000,0.000000,0.000000,0.636042,6.678445


In [23]:
eventos.to_pickle("eventos_mundial2018.pkl")
tabla.to_pickle("tabla_jugadores.pkl")

In [24]:
# 1) ¿Cuadran los goles de Croacia? (14 reales, menos los goles en propia puerta rival)
print(tabla[tabla.equipo == "Croatia"].goles.sum())

# 2) Minutos de Modrić (debería rondar 600-690)
print(tabla[tabla.jugador.str.contains("Modri")][["jugador", "minutos", "progresivos_90"]])

# 3) Top 10 de progresivos por 90
tabla.sort_values("progresivos_90", ascending=False)[["jugador", "equipo", "minutos", "progresivos_90"]].head(10)

12.0
         jugador  minutos  progresivos_90
329  Luka Modrić      706        5.991501


,jugador,equipo,minutos,progresivos_90
288,Jérôme Boateng,Germany,190,12.315789
78,Branislav Ivanović,Serbia,192,11.718750
542,Toni Kroos,Germany,289,11.211073
341,Marcelo Vieira da Silva Júnior,Brazil,298,9.966443
268,Josh Risdon,Australia,282,9.574468
389,Mohamed Abdul Shafy Sayed Abouezeid,Egypt,285,8.842105
544,Trent Sainsbury,Australia,282,8.617021
296,Kasper Schmeichel,Denmark,402,8.507463
586,Yuri Zhirkov,Russia,224,8.437500
382,Miguel Ángel Trauco Saavedra,Peru,283,8.268551


In [25]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

METRICAS = ["pases_90", "progresivos_90", "tiros_90", "xg_90", "regates_90", "recuperaciones_90"]

t = tabla.reset_index(drop=True)
X = StandardScaler().fit_transform(t[METRICAS])
sim = cosine_similarity(X)

def parecidos(nombre, n=10):
    idx = t.index[t.jugador.str.contains(nombre, case=False)]
    if len(idx) == 0:
        print("No encontrado"); return
    i = idx[0]
    res = t.assign(similitud=sim[i]).drop(index=i)
    return (res.sort_values("similitud", ascending=False)
               [["jugador", "equipo", "minutos", "similitud"]].head(n))

print(t.jugador[t.jugador.str.contains("Modri", case=False)].iloc[0])
parecidos("Modri")

Luka Modrić


,jugador,equipo,minutos,similitud
287,Salman Mohammed Al Faraj,Saudi Arabia,283,0.940238
348,Éver Maximiliano David Banega,Argentina,231,0.822264
181,Kevin De Bruyne,Belgium,566,0.807090
131,Héctor Miguel Herrera López,Mexico,378,0.789656
243,Nabil Dirar,Morocco,191,0.776419
298,Shinji Kagawa,Japan,235,0.773274
50,Christian Alberto Cueva Bravo,Peru,270,0.772123
206,Marcelo Vieira da Silva Júnior,Brazil,298,0.768043
313,Toni Kroos,Germany,289,0.767760
2,Abdullah Ibrahim Otayf,Saudi Arabia,252,0.762396


In [31]:
pos = (eventos.dropna(subset=["player", "position"])
       .groupby(["player", "team"]).position
       .agg(lambda s: s.value_counts().index[0])
       .rename("posicion").reset_index()
       .rename(columns={"player": "jugador", "team": "equipo"}))


filtro por posicion y por minutos para eliminar el ruido de los laterales y los jugadores con menos minutos

In [27]:
ROLES = {
    "Goalkeeper": "Portero",
    "Center Back": "Central", "Left Center Back": "Central", "Right Center Back": "Central",
    "Right Back": "Lateral", "Left Back": "Lateral",
    "Right Wing Back": "Lateral", "Left Wing Back": "Lateral",
    "Left Center Midfield": "Centrocampista", "Right Center Midfield": "Centrocampista",
    "Center Midfield": "Centrocampista", "Center Attacking Midfield": "Centrocampista",
    "Right Attacking Midfield": "Centrocampista", "Left Attacking Midfield": "Centrocampista",
    "Right Defensive Midfield": "Centrocampista", "Center Defensive Midfield": "Centrocampista",
    "Left Defensive Midfield": "Centrocampista",
    "Left Midfield": "Extremo", "Right Midfield": "Extremo",
    "Left Wing": "Extremo", "Right Wing": "Extremo",
    "Center Forward": "Delantero", "Right Center Forward": "Delantero",
    "Left Center Forward": "Delantero", "Secondary Striker": "Delantero",
}
pos["rol"] = pos.posicion.map(ROLES)
print(pos.rol.value_counts())

tabla2 = tabla.merge(pos[["jugador", "equipo", "rol"]], on=["jugador", "equipo"], how="left")
print(tabla2.rol.isna().sum(), "jugadores sin rol")

rol
Centrocampista    152
Extremo           125
Central           103
Lateral            95
Delantero          90
Portero            41
Name: count, dtype: int64
0 jugadores sin rol


ahora hago la comparacion de los jugadores por posicion parecidos a modric

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

METRICAS = ["pases_90", "progresivos_90", "tiros_90", "xg_90", "regates_90", "recuperaciones_90"]
MIN_MINUTOS = 270

def parecidos(nombre, n=10, min_minutos=MIN_MINUTOS):
    base = tabla2[tabla2.jugador.str.contains(nombre, case=False)]
    if base.empty:
        print("No encontrado"); return
    jug = base.iloc[0]
    print(f"{jug.jugador} ({jug.equipo}) | rol: {jug.rol} | {jug.minutos:.0f} min")

    grupo = tabla2[(tabla2.rol == jug.rol) & (tabla2.minutos >= min_minutos)].reset_index(drop=True)
    X = StandardScaler().fit_transform(grupo[METRICAS])
    sim = cosine_similarity(X)

    i = grupo.index[grupo.jugador == jug.jugador]
    if len(i) == 0:
        print("El jugador no llega al mínimo de minutos"); return
    i = i[0]
    res = grupo.assign(similitud=sim[i]).drop(index=i)
    return (res.sort_values("similitud", ascending=False)
               [["jugador", "equipo", "minutos", "similitud"]].head(n))

parecidos("Modri")

Luka Modrić (Croatia) | rol: Centrocampista | 706 min


,jugador,equipo,minutos,similitud
52,Salman Mohammed Al Faraj,Saudi Arabia,283,0.898605
22,Héctor Miguel Herrera López,Mexico,378,0.700232
57,Toni Kroos,Germany,289,0.688769
33,Kevin De Bruyne,Belgium,566,0.646954
44,Oghenekaro Etebo,Nigeria,277,0.584749
49,Rodrigo Bentancur Colmán,Uruguay,372,0.555386
12,Christian Alberto Cueva Bravo,Peru,270,0.522474
10,Carlos Henrique Casimiro,Brazil,346,0.521450
4,Andrés Iniesta Luján,Spain,290,0.516596
47,Paul Pogba,France,563,0.494111


ahora el analisis de mas aspectos de modric

In [29]:
tabla2[tabla2.jugador.str.contains("Modri")][["jugador", "minutos", "pases_90", "progresivos_90", "tiros_90", "xg_90", "regates_90", "recuperaciones_90"]]

,jugador,minutos,pases_90,progresivos_90,tiros_90,xg_90,regates_90,recuperaciones_90
197,Luka Modrić,706,54.815864,5.991501,1.274788,0.234568,2.03966,7.13881


aqui comparo a kroos y modric con el mejor de la tabla

In [30]:
nombres = "Modri|Al Faraj|Kroos"
grupo = tabla2[(tabla2.rol == "Centrocampista") & (tabla2.minutos >= 270)]
print(grupo[grupo.jugador.str.contains(nombres)][["jugador", "minutos"] + METRICAS].round(2))

                      jugador  minutos  pases_90  progresivos_90  tiros_90  \
197               Luka Modrić      706     54.82            5.99      1.27   
287  Salman Mohammed Al Faraj      283     72.19            5.41      0.95   
313                Toni Kroos      289     90.93           11.21      4.98   

     xg_90  regates_90  recuperaciones_90  
197   0.23        2.04               7.14  
287   0.31        1.91               7.31  
313   0.22        1.56               7.79  
